In [1]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_pickle('../data/ca1_cleaned.pkl')

# Keep last 2 years (730 days) for fast iteration & clean memory
max_date = df['date'].max()
cutoff_date = max_date - pd.Timedelta(days=730)
df = df[df['date'] >= cutoff_date].sort_values(['id', 'date']).reset_index(drop=True)

print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Working dataset shape: {df.shape}")


Date range: 2014-05-23 to 2016-05-22
Working dataset shape: (2196550, 15)


In [2]:
# 1. Calendar features
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype('int8')
df['day_of_month'] = df['date'].dt.day.astype('int8')
df['month'] = df['date'].dt.month.astype('int8')
df['has_event'] = df['event_name_1'].notna().astype('int8')

# 2. Price features (capturing promotions / price drops)
df['price_max'] = df.groupby('id')['sell_price'].transform('max')
df['price_norm'] = df['sell_price'] / df['price_max']  # < 1.0 indicates a discount
df['price_momentum'] = df['sell_price'] / df.groupby('id')['sell_price'].shift(7)

# 3. Lag & Rolling Demand Features (shifted by 28 days to prevent leakage)
grouped_sales = df.groupby('id')['sales']

df['lag_28'] = grouped_sales.shift(28)
df['lag_35'] = grouped_sales.shift(35)  # Same day of week, 5 weeks ago
df['lag_42'] = grouped_sales.shift(42)  # Same day of week, 6 weeks ago

# Rolling stats on shifted sales
df['rolling_mean_7'] = grouped_sales.transform(lambda x: x.shift(28).rolling(7).mean())
df['rolling_mean_28'] = grouped_sales.transform(lambda x: x.shift(28).rolling(28).mean())
df['rolling_std_28'] = grouped_sales.transform(lambda x: x.shift(28).rolling(28).std())

# Drop initial rows where lags are NaN
df = df.dropna(subset=['lag_42', 'rolling_mean_28']).reset_index(drop=True)
print(f"Dataset shape after feature engineering: {df.shape}")


Dataset shape after feature engineering: (2028855, 29)


In [3]:
# Hold out the final 28 days as our validation set
val_start_date = df['date'].max() - pd.Timedelta(days=27)

train_df = df[df['date'] < val_start_date].copy()
val_df = df[df['date'] >= val_start_date].copy()

print(f"Train rows: {len(train_df):,} | Validation rows: {len(val_df):,} ({val_df['date'].nunique()} days)")

# Define WMAPE metric
def wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

# Baseline 1: Seasonal Naive (Predict same day of week from 28 days ago)
baseline_naive_wmape = wmape(val_df['sales'], val_df['lag_28'])

# Baseline 2: 28-Day Rolling Mean
baseline_ma_wmape = wmape(val_df['sales'], val_df['rolling_mean_28'])

print(f"Baseline 1 (Seasonal Naive - Lag 28) WMAPE: {baseline_naive_wmape:.2f}%")
print(f"Baseline 2 (28-Day Moving Average) WMAPE:   {baseline_ma_wmape:.2f}%")

# Save feature-engineered dataset for ML modeling
df.to_pickle('../data/ca1_features.pkl')
print("Saved ca1_features.pkl successfully!")


Train rows: 1,943,483 | Validation rows: 85,372 (28 days)
Baseline 1 (Seasonal Naive - Lag 28) WMAPE: 89.82%
Baseline 2 (28-Day Moving Average) WMAPE:   76.74%
Saved ca1_features.pkl successfully!
